## Problem Statement

Credit card fraud detection is essentially an anomaly detection problem. We define an anomaly as a transaction that appears significantly different from how real transactions typically behave, not just in one feature but across all 30 features at the same time. 

We handle this as a semi-supervised problem. This means we train only on genuine transactions and use labels only during evaluation. This choice is intentional. A supervised approach would require showing the model labeled fraud examples during training, which poses problems for two reasons. First, fraud accounts for only 0.17% of transactions. Second, fraud patterns change over time, so past examples may not represent future attacks. An unsupervised approach would ignore our labels altogether, wasting the valuable ground truth we have for evaluation.

The semi-supervised approach makes the most sense for this dataset. Legit transactions are plentiful and consistent, providing a solid base to learn normal behavior. Instead of memorizing what fraud looks like, we learn what normal looks like. Anything that deviates from that is flagged as suspicious.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_recall_curve, roc_curve, auc,
                             confusion_matrix, classification_report,
                             f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

df = pd.read_csv('../data/creditcard.csv')

legit = df[df['Class'] == 0]
fraud = df[df['Class'] == 1]

legit_train, legit_test = train_test_split(legit, test_size=0.2, random_state=5)
test_set = pd.concat([legit_test, fraud]).sample(frac=1, random_state=5)


scaler = StandardScaler()
legit_train = legit_train.copy()
test_set = test_set.copy()

legit_train[['Amount', 'Time']] = scaler.fit_transform(legit_train[['Amount', 'Time']])
test_set[['Amount', 'Time']] = scaler.transform(test_set[['Amount', 'Time']])

X_train = legit_train.drop('Class', axis=1).values
X_test = test_set.drop('Class', axis=1).values
y_test = test_set['Class'].values

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape} ")

Training set: (227452, 30)
Test set: (57355, 30) 
